# QC confound 03 — Does the result depend on Pal's per-sample thresholds?

Which cells count as "discarded" is set by per-sample thresholds, and Pal lowered the
gene threshold for low-coverage samples. That makes noise membership itself a
sample-level signal that could track condition. Here the published definition is
replaced by rules that treat every specimen identically:

| Rule | Definition |
|---|---|
| `published` | Pal's per-sample thresholds (as in the poster). |
| `uniform` | One study-wide rule: the most common value of each published threshold, applied to every specimen. |
| `mad` | One adaptive procedure: per specimen, flag cells more than 3 MADs out on log genes (both tails), log library size (upper), or mito fraction (upper), mirroring Pal's four flags. |

For each rule the core arms from notebook 01 are re-run: `noise_pbs` (published
pipeline), `retained_pbs_matched` (negative control), `noise_qc_rank` and `retained_qc`.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from signals_in_the_noise.analysis.noise_phenotypes import ISCB_SCS_PBS_THRESHOLDS
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.preprocessing.gse161529 import GSE161529
from signals_in_the_noise.preprocessing.specimens import Cohort
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

POSITIVE, NEGATIVE = "ER+ tumour", "Normal"
SEED = 0
OUTPUT_DIR = get_data_path("processed") / "qc_confound"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
gse = GSE161529()
cohort = Cohort.from_objects(gse.objects.values(), [POSITIVE, NEGATIVE])
cohort.overview().groupby("condition").agg(specimens=("n_cells", "size"), cells=("n_cells", "sum"))

In [ ]:
from signals_in_the_noise.preprocessing.qc import (
    QcThresholds,
    mad_thresholds,
    modal_thresholds,
    relabel_noise,
)

N_MADS = 3.0
published = {sid: QcThresholds.from_uns(adata.uns) for sid, adata in cohort.specimens.items()}
uniform = modal_thresholds(published.values())
print("uniform rule:", uniform)

published_table = pd.DataFrame({sid: vars(t) for sid, t in published.items()}).T
published_table.insert(0, "condition", pd.Series(cohort.conditions))
published_table

In [ ]:
RULES = {"published": "is_noise", "uniform": "is_noise_uniform", "mad": "is_noise_mad"}

rule_obs = relabel_noise(cohort.obs, lambda obs: uniform, RULES["uniform"])
rule_obs = relabel_noise(rule_obs, lambda obs: mad_thresholds(obs, n_mads=N_MADS), RULES["mad"])

## 1. How much does each rule change the discarded population?

`noise_fraction` per specimen under each rule, its association with condition, and the
share of cells whose noise flag differs from the published one.

In [ ]:
from signals_in_the_noise.analysis.confounders import association_table, covariates_from_obs

burden = pd.DataFrame(
    {
        rule: covariates_from_obs(rule_obs, cohort.conditions, col)["noise_fraction"]
        for rule, col in RULES.items()
    }
)
burden.insert(0, "condition", pd.Series(cohort.conditions))
for rule, col in RULES.items():
    burden[f"changed_vs_published_{rule}"] = pd.Series(
        {sid: float((obs[col] != obs["is_noise"]).mean()) for sid, obs in rule_obs.items()}
    )
display(burden.round(3))
display(
    association_table(burden, positive=POSITIVE, negative=NEGATIVE, columns=list(RULES)).round(4)
)

## 2. Core arms under each rule

In [ ]:
from signals_in_the_noise.modeling.datasets import CORE_ARM_NAMES, standard_arms
from signals_in_the_noise.modeling.experiments import (
    PUBLISHED_PROTOCOL,
    InsufficientSpecimensError,
    run_arm_suite,
)

N_PERMUTATIONS = 200

results = {}
for rule, column in RULES.items():
    try:
        results[rule] = run_arm_suite(
            rule_obs,
            cohort.conditions,
            standard_arms(ISCB_SCS_PBS_THRESHOLDS, names=CORE_ARM_NAMES),
            positive_label=POSITIVE,
            negative_label=NEGATIVE,
            patients=cohort.patients,
            noise_column=column,
            n_permutations=N_PERMUTATIONS,
            seed=SEED,
            **PUBLISHED_PROTOCOL,
        )
    except InsufficientSpecimensError as error:
        print(f"{rule}: skipped ({error})")
        continue
    print(f"{rule}: {len(results[rule].specimens)} specimens included")
assert results, "No noise definition left enough specimens to compare conditions."

In [ ]:
summary = pd.concat({rule: r.summary() for rule, r in results.items()}, names=["rule", "arm"])
display(
    summary[
        [
            "n_specimens",
            "specimen_auc",
            "permutation_null_mean",
            "permutation_p",
            "draws_mean_cell_accuracy",
            "draws_mean_specimen_auc",
        ]
    ].round(3)
)
paired = pd.concat(
    {rule: r.paired_vs_reference("cell_accuracy") for rule, r in results.items()},
    names=["rule", "arm"],
)
display(paired.round(3))

ax = sns.pointplot(
    data=summary.reset_index(), x="rule", y="specimen_auc", hue="arm", dodge=0.3, linestyle="none"
)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1)
ax.set_title("Specimen AUC by noise definition")
plt.show()

In [ ]:
burden.to_csv(OUTPUT_DIR / "03_noise_burden_by_rule.csv")
summary.to_csv(OUTPUT_DIR / "03_summary_by_rule.csv")
paired.to_csv(OUTPUT_DIR / "03_paired_by_rule.csv")

## Reading the results

| Observation | Meaning |
|---|---|
| `noise_pbs` stays significant and above `retained_pbs_matched` under `uniform` and `mad` | The result does not depend on Pal's per-sample thresholds or the low-coverage adjustment. |
| `noise_pbs` only works under `published` | The per-sample threshold choice — itself tied to coverage — is carrying the signal. |
| `noise_fraction` separates conditions under every rule | Conditions genuinely differ in how many cells look low quality; report it as a finding, not as PBS biology. |